<a href="https://colab.research.google.com/github/RsSubhamMohanty/The-Reasoners-Educational-Content-Generator-AI-Agent-Development-Project-Dual-Track-Version-/blob/main/Educational_Agent_Week6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install groq

In [ ]:
!pip install langchain-text-splitters
!pip install sentence-transformers

In [ ]:
from groq import Groq

In [ ]:
client = Groq(api_key="")

In [ ]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role":"user","content":"Explain Artificial Intelligence in 3 simple lines"}]
)

print(response.choices[0].message.content)

In [ ]:
def generate_quiz(study_text):

    prompt = f"""
    You are an AI Study Assistant.

    From the study material below create 5 multiple choice questions.

    Study Material:
    {study_text}

    Format:

    Question
    A)
    B)
    C)
    D)
    Correct Answer
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role":"user","content":prompt}]
    )

    return response.choices[0].message.content

In [ ]:
study_notes = """
Machine Learning is a branch of Artificial Intelligence that allows computers
to learn patterns from data and improve performance without being explicitly programmed.
"""

In [ ]:
quiz = generate_quiz(study_notes)

print(quiz)

In [ ]:
!pip install PyPDF2

In [ ]:
import PyPDF2

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
pdf_text = ""

for file_name in uploaded.keys():
    pdf_reader = PyPDF2.PdfReader(file_name)

    for page in pdf_reader.pages:
        pdf_text += page.extract_text()

print("PDF text extracted successfully")

In [ ]:
print(pdf_text[:1000])

In [ ]:
quiz = generate_quiz(pdf_text)

print(quiz)

In [ ]:
!pip install chromadb
!pip install sentence-transformers
!pip install langchain-community
!pip install langchain-text-splitters

In [ ]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_text(pdf_text)

print("Total chunks:", len(chunks))

In [ ]:
embeddings = HuggingFaceEmbeddings()

In [ ]:
vector_db = Chroma.from_texts(
    chunks,
    embeddings
)

print("Vector database created")

In [ ]:
query = "Create quiz questions from this study material"

docs = vector_db.similarity_search(query)

context = docs[0].page_content

print(context[:500])

In [ ]:
prompt = f"""
Create 5 multiple choice questions from the following study material.

{context}
"""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role":"user","content":prompt}]
)

print(response.choices[0].message.content)

In [ ]:
!pip install langchain
!pip install langchain-community
!pip install duckduckgo-search

In [ ]:
!pip install langchain-core
!pip install langchain
!pip install langchain-community

In [ ]:
def run_agent(task, context):

    if task == "quiz":
        return generate_quiz(context)

    elif task == "flashcards":
        return generate_flashcards(context)

    elif task == "summary":
        return generate_summary(context)

    else:
        return "Task not recognized. Use: quiz, flashcards, or summary."

In [ ]:
result = run_agent("quiz", context)

print(result)

In [ ]:
def generate_flashcards(context):

    prompt = f"""
    Create 5 study flashcards from the following material.

    Format:
    Question:
    Answer:

    Material:
    {context}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
def generate_summary(context):

    prompt = f"""
    Summarize the following study material in clear bullet points.

    Material:
    {context}
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
result = run_agent("flashcards", context)

print(result)

In [ ]:
result = run_agent("summary", context)

print(result)

In [ ]:
!pip install streamlit
!pip install pyngrok

In [ ]:
pip install fpdf

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os
os.listdir()

In [ ]:
!mv "Home.png.png" home.png

In [ ]:
!pip install gtts

In [ ]:
!pip install youtube-search-python matplotlib

In [ ]:
!rm study.db

In [ ]:
%%writefile app.py

import streamlit as st
import PyPDF2
from groq import Groq
import sqlite3
from datetime import datetime
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pandas as pd
from datetime import datetime

# ===== PAGE CONFIG =====
st.set_page_config(layout="wide")

# ===== UI ENHANCEMENT (ONLY ADDED) =====
st.markdown("""
<style>

/* Background */
.stApp {
    background: linear-gradient(135deg, #0f172a, #1e293b);
    color: white;
}

/* Cards */
.block-container {
    padding-top: 2rem;
}

/* Glass Cards */
div[data-testid="stVerticalBlock"] > div {
    background: rgba(255, 255, 255, 0.05);
    backdrop-filter: blur(12px);
    border-radius: 16px;
    padding: 18px;
    margin-bottom: 12px;
    border: 1px solid rgba(255,255,255,0.08);
}

/* Buttons */
button {
    border-radius: 14px !important;
    background: linear-gradient(135deg, #6366f1, #8b5cf6) !important;
    color: white !important;
    border: none !important;
    transition: 0.2s;
}

button:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 20px rgba(99,102,241,0.4);
}

/* Input */
input, textarea {
    border-radius: 10px !important;
    border: 1px solid #334155 !important;
    background: #0f172a !important;
    color: white !important;
}

/* Selectbox */
div[data-baseweb="select"] {
    background: #0f172a !important;
    border-radius: 10px !important;
}

/* Headings */
h1 {
    font-size: 38px;
    font-weight: 700;
}

h2,h3 {
    color: #c7d2fe;
}

/* Success box */
.stAlert {
    border-radius: 12px;
}

/* Divider */
hr {
    border: 1px solid rgba(255,255,255,0.08);
}

</style>
""", unsafe_allow_html=True)

# ===== HEADER =====
st.markdown("""
<div style="
    background: linear-gradient(135deg, #6366f1, #8b5cf6);
    padding: 20px;
    border-radius: 16px;
    text-align: center;
    font-size: 28px;
    font-weight: bold;
    color: white;
    margin-bottom: 20px;
">
🚀 AI Study Assistant
</div>
""", unsafe_allow_html=True)

st.markdown("""
<div style="text-align:center; color:#94a3b8; margin-bottom:20px;">
"Stay consistent, success will follow 💡"
</div>
""", unsafe_allow_html=True)

# ===== GROQ =====
client = Groq(api_key="")

# ===== DATABASE =====
conn = sqlite3.connect("study.db", check_same_thread=False)
c = conn.cursor()

c.execute("""
CREATE TABLE IF NOT EXISTS history (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    type TEXT,
    topic TEXT,
    date TEXT
)
""")

c.execute("""
CREATE TABLE IF NOT EXISTS schedule (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    topic TEXT,
    study_date TEXT,
    study_time TEXT,
    task_type TEXT,
    status TEXT
)
""")

conn.commit()

# ===== SESSION =====
for key in ["quiz_result","flashcard_result","summary_result","audio_result","topics","selected_topic","active_tab"]:
    if key not in st.session_state:
        st.session_state[key] = "" if "result" in key else []

if st.session_state.active_tab == []:
    st.session_state.active_tab = "quiz"

# ===== FILE =====
uploaded_file = st.file_uploader("📄 Upload PDF", type=["pdf"])

# ===== FUNCTIONS =====
def extract_text(file):
    reader = PyPDF2.PdfReader(file)
    text = ""
    for page in reader.pages:
        if page.extract_text():
            text += page.extract_text()
    return text

def ask_llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role":"user","content":prompt}]
    )
    return response.choices[0].message.content

def detect_topics(text):
    prompt = f"""
    Extract main topics from the content.
    Return only comma-separated topics.

    {text[:3000]}
    """
    topics = ask_llm(prompt)
    return [t.strip() for t in topics.split(",") if t.strip()]

def get_youtube_videos(topic):
    return [f"https://www.youtube.com/results?search_query={topic}+education"]

# ===== INPUT =====
num_questions = st.number_input("Number of Questions",1,20,5)

# ===== BUTTONS =====
col1, col2, col3, col4 = st.columns([1,1,1,1])

with col1:
    quiz = st.button("📝 Quiz")

with col2:
    flashcard = st.button("🧠 Flashcard")

with col3:
    summary = st.button("📌 Summary")

with col4:
    audio = st.button("🎧 Audio")

# ===== MAIN =====
if uploaded_file:

    context = extract_text(uploaded_file)

    if not st.session_state.topics:
        st.session_state.topics = detect_topics(context)

    topics = st.session_state.topics

    if not st.session_state.selected_topic:
        st.session_state.selected_topic = topics[0]

    selected_topic = st.selectbox(
        "Select Topic",
        topics,
        index=topics.index(st.session_state.selected_topic)
    )

    st.session_state.selected_topic = selected_topic
    st.success(f"Selected Topic: {selected_topic}")

    # ===== FIXED QUIZ =====
    if quiz:
        st.session_state.active_tab = "quiz"

        st.session_state.quiz_result = ask_llm(f"""
        Create {num_questions} MCQs from {selected_topic}:
        {context}
        """)

        c.execute("INSERT INTO history VALUES (NULL,?,?,?)",
                  ("Quiz",selected_topic,str(datetime.now())))
        conn.commit()

    if flashcard:
        st.session_state.active_tab = "flashcard"

        if not st.session_state.flashcard_result:
            st.session_state.flashcard_result = ask_llm(f"""
            Create {num_questions} flashcards from {selected_topic}:

            {context}
            """)

            c.execute("INSERT INTO history VALUES (NULL,?,?,?)",
                      ("Flashcard", selected_topic, str(datetime.now())))
            conn.commit()

    if summary:
        st.session_state.active_tab = "summary"

        if not st.session_state.summary_result:
            st.session_state.summary_result = ask_llm(f"""
            Summarize {selected_topic}:

            {context}
            """)

            c.execute("INSERT INTO history VALUES (NULL,?,?,?)",
                      ("Summary", selected_topic, str(datetime.now())))
            conn.commit()

    if audio:
              st.session_state.active_tab = "audio"

              if not st.session_state.audio_result:
                  st.session_state.audio_result = ask_llm(f"""
                  Explain {selected_topic} simply:

                  {context}
                  """)

              # 🔊 Audio generation
              from gtts import gTTS

              if st.session_state.audio_result:
                  tts = gTTS(st.session_state.audio_result)
                  tts.save("audio.mp3")

                  # ✅ SHOW AUDIO ONLY ONCE
                  st.audio("audio.mp3")

    # ===== DISPLAY =====
    content = ""

    if st.session_state.active_tab == "quiz":
        st.markdown("## 📝 Quiz")
        content = st.session_state.quiz_result

    elif st.session_state.active_tab == "flashcard":
        st.markdown("## 🧠 Flashcards")
        content = st.session_state.flashcard_result

    elif st.session_state.active_tab == "summary":
        st.markdown("## 📌 Summary")
        content = st.session_state.summary_result

    elif st.session_state.active_tab == "audio":
        st.markdown("## 🎧 Audio Explanation")
        content = st.session_state.audio_result

    if content:
        st.write(content)

    # ===== STUDY PLANNER (RESTORED) =====
    st.markdown("---")
    st.markdown("## 📅 Study Planner")

    # SAME TOPIC SELECTOR
    planner_topic = st.selectbox("Select Topic for Study Plan", topics)

    study_date = st.date_input("Select study date")

    study_time = st.time_input("⏱️ Select Time")

    task_type = st.selectbox(
        "🎯 Task Type",
        ["Read", "Practice", "Revise", "Quiz"]
    )

    if st.button("➕ Add to Study Plan"):

        c.execute(
            "INSERT INTO schedule (topic, study_date, study_time, task_type, status) VALUES (?, ?, ?, ?, ?)",
            (planner_topic, str(study_date), str(study_time), task_type, "Pending")
        )
        conn.commit()

        st.success(f"Added '{planner_topic}' to your study plan")

    st.markdown("### 📌 Your Study Schedule")

    plans = c.execute("SELECT id, topic, study_date, study_time, task_type, status FROM schedule").fetchall()

    for plan in plans:
        col1, col2, col3, col4, col5, col6 = st.columns([2,2,2,2,2,1])

        col1.write(plan[1])
        col2.write(plan[2])
        col3.write(plan[3])
        col4.write(plan[4])
        col5.write(plan[5])

        if col6.button("✔", key=plan[0]):
            c.execute("UPDATE schedule SET status='Done' WHERE id=?", (plan[0],))
            conn.commit()
            st.rerun()

    # ===== YOUTUBE =====
    st.markdown("## 🎬 YouTube Learning")
    st.caption("Watch related videos for this topic")
    if st.button("🎥 Learn from YouTube"):
        links = get_youtube_videos(selected_topic)
        for link in links:
            st.markdown(f"[Watch]({link})")

    # ===== ANALYTICS  =====
    # 📊 ANALYTICS
    st.markdown("## 📊 Study Analytics")

    view = st.radio(
        "Select View",
        ["Daily", "Monthly"],
        horizontal=True
    )

    # 🔹 FETCH DATA
    data = c.execute("SELECT date FROM history").fetchall()

    if data:

        df = pd.DataFrame(data, columns=["date"])
        df["date"] = pd.to_datetime(df["date"])

        if view == "Daily":
            df["day"] = df["date"].dt.date
            grouped = df.groupby("day").size()

            x = list(range(1, len(grouped)+1))
            y = grouped.values

        else:
            # 🔥 MONTHLY (REAL DAYS 1–30/31)
            current_month = datetime.now().month
            current_year = datetime.now().year

            df = df[
                (df["date"].dt.month == current_month) &
                (df["date"].dt.year == current_year)
            ]

            df["day"] = df["date"].dt.day
            grouped = df.groupby("day").size()

            # FULL DAYS (1–30/31)
            days_in_month = 31
            x = list(range(1, days_in_month + 1))
            y = [grouped.get(day, 0) for day in x]

        # 🎯 CONVERT TO HOURS LABEL
        y_labels = [f"{val} hr" if val > 0 else "0" for val in y]

        # 📈 GRAPH
        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=x,
            y=y,
            mode='lines+markers',
            line=dict(width=3, color="#4facfe"),   # blue
            marker=dict(size=6, color="#a855f7"),  # purple
            hovertemplate="Day %{x}<br>Study: %{y} hr<extra></extra>"
        ))

        fig.update_layout(
            height=320,
            template="plotly_dark",
            margin=dict(l=10, r=10, t=20, b=10),
            xaxis_title="Days",
            yaxis_title="Study Hours",
            xaxis=dict(tickmode='linear'),
            yaxis=dict(tickmode='linear')
        )

        # 🎯 LAYOUT
        col1, col2 = st.columns([3, 1])

        with col1:
            st.plotly_chart(fig, use_container_width=True)

        with col2:
            st.markdown("### 🔥 Streak")

            unique_days = sorted(set(df["date"].dt.date))
            streak = 1

            for i in range(1, len(unique_days)):
                if (unique_days[i] - unique_days[i-1]).days == 1:
                    streak += 1

            st.markdown(f"🔥 **{streak} days**")

    else:
        st.info("No study data yet 📚")

In [ ]:
from pyngrok import ngrok
ngrok.kill()

In [ ]:
import subprocess, time
process = subprocess.Popen(["streamlit", "run", "app.py"])
time.sleep(5)

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3AsXDHdlpj8GJGHeEysjPOTTj0R_6WPAXSDiNCyZSZNXm6cWW")

public_url = ngrok.connect(8501)
print("OPEN THIS LINK:", public_url)